# 01 - Engine and Session Smoke

Use this notebook to validate engine detection and basic session lifecycle on a Windows host.

## Expected
- Engine should be `COM` on a licensed Windows machine with PowerPoint installed.
- Create, list, state, save, close should all succeed.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python").exists():
    if (REPO_ROOT.parent / "python").exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError("Run notebook from repo root or notebooks/ directory")

PYTHON_ROOT = REPO_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))


def build_service():
    from service import PowerPointService

    return PowerPointService


ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook-tests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Artifact dir:", ARTIFACT_DIR)

In [ ]:
service = build_service()()
engine_info = service.dispatch("pptx_get_engine_info", {})
pprint(engine_info)

if engine_info.get("engine") != "COM":
    print("WARNING: COM is not active. On Windows host this should normally be COM.")

In [ ]:
create_result = service.dispatch("pptx_create_presentation", {"width": "10in", "height": "5.625in"})
presentation_id = create_result["presentation_id"]
print("Presentation ID:", presentation_id)
print("Create keys:", sorted(create_result.keys()))

list_result = service.dispatch("pptx_list_open_presentations", {})
state_result = service.dispatch("pptx_get_presentation_state", {"presentation_id": presentation_id})
print("Open sessions:", len(list_result.get("presentations", [])))
print("Slide count:", state_result.get("slide_count"))

In [ ]:
output_path = (ARTIFACT_DIR / "nb01_engine_session_output.pptx").resolve()
save_result = service.dispatch(
    "pptx_save_presentation",
    {
        "presentation_id": presentation_id,
        "output_path": str(output_path),
    },
)
print("Saved path:", save_result.get("saved_path"))

close_result = service.dispatch("pptx_close_presentation", {"presentation_id": presentation_id})
print("Closed:", close_result.get("closed"))

service.shutdown()
print("Shutdown complete")